# ForecastClean — Chronos run (Colab / Kaggle)

Runs the Chronos foundation models on the same 9 series as the classical baselines.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (CPU also works, just slower).  
**Status:** the adapter is unit-tested with stand-in models only — *this notebook is the first real test.* If a cell fails, copy the full error back to Claude.

The repo must be reachable by `git clone` (public repo, or use a token).

In [ ]:
# 1) get the code and install (chronos-forecasting pulls in torch/transformers if missing)
!git clone https://github.com/mirzayasirabdullahbaig07/ForecastClean.git
%cd ForecastClean
!pip -q install -e . chronos-forecasting
!pip show chronos-forecasting | head -3
import torch; print('cuda available:', torch.cuda.is_available())

In [ ]:
# 2) raw data is NOT in git (licences) -> download it here
!forecastclean download
!cat data/MANIFEST.csv | cut -d, -f1,5,7,8

In [ ]:
# 3) SMOKE TEST (a few minutes): one tiny model, two series, few windows
!forecastclean batch --only wiki_Python ecb_EURUSD --chronos amazon/chronos-bolt-tiny --max-origins 10 --out results/smoke

In [ ]:
# 4) FULL RUN: Bolt tiny/small + Chronos-2 on all series (CPU may take an hour or more)
!forecastclean batch --chronos amazon/chronos-bolt-tiny amazon/chronos-bolt-small amazon/chronos-2 --out results/chronos

In [ ]:
# 4b) THE MAIN ANALYSIS (gap A'): only windows AFTER the latest model release date + horizon sweep
!forecastclean batch --chronos amazon/chronos-bolt-small amazon/chronos-2 --release-cutoff --post-only --horizon-sweep --max-origins 50 --out results/chronos_post

In [ ]:
# 4c) RQ2 CONTAMINATION GAP: each model scored before vs after ITS OWN release date, compared with the classical models
!forecastclean gap --chronos amazon/chronos-bolt-small amazon/chronos-2 --max-origins 60 --out results/gap
import pandas as pd
print(pd.read_csv('results/gap/gap_summary.csv').round(3).to_string())


In [ ]:
# 4d) TUNED BASELINES: Chronos vs AutoETS / AutoTheta (+AutoARIMA on non-seasonal series), post-release windows only.
# The hourly series is left out here (tuned seasonal fits are too slow); say so in the paper.
!pip install -q statsforecast
!forecastclean batch --chronos amazon/chronos-bolt-small amazon/chronos-2 --release-cutoff --post-only --max-origins 30 --tuned --only neso_demand_daily ecb_EURUSD ecb_EURGBP ecb_EURJPY ecb_EURCHF ecb_EURCAD ecb_EURAUD ecb_EURSEK ecb_EURNOK ecb_EURCNY ecb_EURINR wiki_Python wiki_Christmas wiki_Easter wiki_Halloween wiki_Thanksgiving wiki_Wimbledon wiki_SuperBowl wiki_Bitcoin wiki_Cricket wiki_Weather wiki_Ramadan wiki_MachineLearning wiki_TaylorSwift wiki_Eurovision --out results/chronos_post_tuned
import pandas as pd
s = pd.read_csv('results/chronos_post_tuned/summary.csv')
print(s.pivot_table(index='series', columns='model', values='rel_mase').round(2).to_string())


In [ ]:
# 4e) TIMESFM SMOKE TEST (a few minutes) — run this BEFORE 4f/4g. TimesFM adapter is not yet verified on real weights.
!pip install -q "timesfm[torch]" || (git clone -q https://github.com/google-research/timesfm.git /content/timesfm && pip install -q -e "/content/timesfm[torch]")
!python -c "import timesfm; print(timesfm.TimesFM_2p5_200M_torch)"
!forecastclean batch --chronos google/timesfm-2.5-200m-pytorch --only ecb_EURUSD wiki_Python --post-only --release-cutoff --max-origins 5 --out results/timesfm_smoke


In [ ]:
# 4f) GAP RE-RUN WITH PER-WINDOW OUTPUT (enables the window-level block bootstrap). ~1.5 h on a T4 (three models); run 4e (TimesFM smoke test) FIRST — if it fails, delete the TimesFM id from this command
!git log -1 --format="%h %s"
!forecastclean gap --chronos amazon/chronos-bolt-small amazon/chronos-2 google/timesfm-2.5-200m-pytorch --max-origins 40 --drop arima --out results/gap_blocks
!ls -la results/gap_blocks   # must contain gap_origins.csv
!forecastclean robust --dir results/gap_blocks --block 5


In [ ]:
# 4g) RQ1 FULL RE-RUN WITH TIMESFM + TUNED BASELINES (~1 h). Also SAVES the result CSVs (the earlier tuned run printed only a table).
!forecastclean batch --chronos amazon/chronos-bolt-small amazon/chronos-2 google/timesfm-2.5-200m-pytorch --release-cutoff --post-only --max-origins 30 --tuned --only neso_demand_daily ecb_EURUSD ecb_EURGBP ecb_EURJPY ecb_EURCHF ecb_EURCAD ecb_EURAUD ecb_EURSEK ecb_EURNOK ecb_EURCNY ecb_EURINR wiki_Python wiki_Christmas wiki_Easter wiki_Halloween wiki_Thanksgiving wiki_Wimbledon wiki_SuperBowl wiki_Bitcoin wiki_Cricket wiki_Weather wiki_Ramadan wiki_MachineLearning wiki_TaylorSwift wiki_Eurovision --out results/chronos_post_tuned3
!ls -la results/chronos_post_tuned3


In [ ]:
# 5) show + download results
import pandas as pd
s = pd.read_csv('results/chronos_post/summary.csv')
print(s.pivot_table(index='series', columns='model', values='rel_mase').round(2).to_string())
!zip -qr chronos_results.zip results data/MANIFEST.csv
from google.colab import files
files.download('chronos_results.zip')   # on Kaggle: the zip appears under /kaggle/working

## After the run
Save `chronos_results.zip`, unzip into your local `ForecastClean` folder and tell Claude. Do **not** commit `data/raw/` (licences). Record versions printed in step 1 in `EXPERIMENT_LOG.md`.